## Этап 1. Загрузка исходных данных

На данном этапе были загружены исходные датасеты `cup_it_example_src_A.csv` и `cup_it_example_src_B.csv`.
Работа проводилассь с копиями данных, чтобы сохранить исходные файлы нетронутыми.
Это позволит при необходимости перезапустить очистку с нуля.

**Начальное количество записей:**
- А: 171 454
- Б: 161 076

Для логирования процесса очистки в копии добавлены служебные поля:
- `status` — статус обработки (clean/repaired/flagged/dropped)
- `quality_flags` — флаги проблем (например, small_building, very_tall)
- `drop_reason` — причина удаления (если объект был удалён)

In [10]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from pathlib import Path

In [11]:
DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/interim")
SOURCE_A_PATH = DATA_DIR / "cup_it_example_src_A.csv"
SOURCE_B_PATH = DATA_DIR / "cup_it_example_src_B.csv"

In [12]:
source_A = pd.read_csv(SOURCE_A_PATH, low_memory=False)
source_B = pd.read_csv(SOURCE_B_PATH, low_memory=False)

df_a = source_A.copy()
df_b = source_B.copy()

initial_count_A = len(df_a)
initial_count_B = len(df_b)

print(f"\nИсходные данные:")
print(f"  А: {initial_count_A} записей")
print(f"  Б: {initial_count_B} записей")


Исходные данные:
  А: 171454 записей
  Б: 161076 записей


In [13]:
# Для датасета А
df_a['status'] = 'clean'           # пока все чистые
df_a['quality_flags'] = ''          # пока пусто
df_a['drop_reason'] = ''            # пока пусто

# Для датасета Б
df_b['status'] = 'clean'
df_b['quality_flags'] = ''
df_b['drop_reason'] = ''

print("Добавлены поля: status, quality_flags, drop_reason")
print(f"А: {df_a.columns.tolist()}")
print(f"B: {df_b.columns.tolist()}")

Добавлены поля: status, quality_flags, drop_reason
А: ['Unnamed: 0', 'id', 'title', 'tags', 'geometry', 'area_sq_m', 'gkh_address', 'gkh_floor_count_min', 'gkh_floor_count_max', 'status', 'quality_flags', 'drop_reason']
B: ['Unnamed: 0', 'subject', 'district', 'type', 'locality', 'type_street', 'name_street', 'number', 'letter', 'fraction', 'housing', 'building', 'purpose_of_building', 'stairs', 'avg_floor_height', 'height', 'wkt', 'id', 'status', 'quality_flags', 'drop_reason']


### Удаление мусорного столбца

В исходных данных присутствовал столбец `Unnamed: 0`, который является техническим индексом и не несёт полезной информации. Удаляем его из копий обоих датасетов.

In [14]:
if 'Unnamed: 0' in df_a.columns:
    df_a = df_a.drop(columns=['Unnamed: 0'])


if 'Unnamed: 0' in df_b.columns:
    df_b = df_b.drop(columns=['Unnamed: 0'])
print(f"А: {df_a.columns.tolist()}")
print(f"B: {df_b.columns.tolist()}")

А: ['id', 'title', 'tags', 'geometry', 'area_sq_m', 'gkh_address', 'gkh_floor_count_min', 'gkh_floor_count_max', 'status', 'quality_flags', 'drop_reason']
B: ['subject', 'district', 'type', 'locality', 'type_street', 'name_street', 'number', 'letter', 'fraction', 'housing', 'building', 'purpose_of_building', 'stairs', 'avg_floor_height', 'height', 'wkt', 'id', 'status', 'quality_flags', 'drop_reason']


### Словарь для статистики

Для учёта количества удалённых и исправленных объектов создаётся словарь `stats`, который будет обновляться на каждом этапе очистки. Это позволит в конце сформировать итоговую таблицу с детальной статистикой.

Счётчики:
- `broken_wkt` — битые WKT-строки (не распарсились)
- `empty_geometry` — пустые геометрии
- `repaired_geometry` — исправленная геометрия
- `zero_area_dropped` — объекты с нулевой площадью
- `small_utility_dropped` — маленькие объекты (<15 м²)
- `duplicate_dropped` — дубликаты
- `records_kept` — количество записей после каждого этапа

In [15]:
# словарь для статистики (счетчики)
stats = {
    'source': ['A', 'B'],
    'initial_count': [len(df_a), len(df_b)],           # исходное количество
    'broken_wkt': [0, 0],                              # битый WKT
    'empty_geometry': [0, 0],                         # пустая геометрия
    'repaired_geometry': [0, 0],                      # исправленная геометрия
    'zero_area_dropped': [0, 0],                      # нулевая площадь
    'small_utility_dropped': [0, 0],                  # маленькие объекты
    'duplicate_dropped': [0, 0],                      # дубликаты
    'records_kept': [len(df_a), len(df_b)]            # пока равно initial_count
}
print(pd.DataFrame(stats))

  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  
0                  0                      0                  0        171454  
1                  0                      0                  0        161076  


## Этап 2. Преобразование WKT в геометрию

Исходные данные хранят геометрию зданий в текстовом формате WKT.
Для пространственного анализа (расчёт площади, поиск пересечений) необходимо преобразовать эти строки в объекты `shapely.geometry`.

Функция `safe_wkt` пытается распарсить строку WKT. Если преобразование не удаётся (битый WKT), объект помечается как `dropped` с причиной `broken_wkt` и удаляется из дальнейшей обработки.

**Результат:**
- Все строки успешно распарсились (100%)
- Битых WKT не обнаружено
- Геометрия готова для дальнейшего анализа

In [16]:
def safe_wkt(x):
    try:
        return wkt.loads(x) if pd.notna(x) else None
    except:
        return None

# Преобразование геометрии
df_a['geometry'] = df_a['geometry'].apply(safe_wkt)
df_b['geometry'] = df_b['wkt'].apply(safe_wkt)

# Считаем, сколько успешно преобразовалось
a_success = df_a['geometry'].notna().sum()
b_success = df_b['geometry'].notna().sum()
print(f"A: успешно {a_success} из {len(df_a)}")
print(f"B: успешно {b_success} из {len(df_b)}")

# Находим битые
a_broken = df_a[df_a['geometry'].isna()].index.tolist()
b_broken = df_b[df_b['geometry'].isna()].index.tolist()
print(f"\nБитых WKT в А: {len(a_broken)}")
print(f"Битых WKT в Б: {len(b_broken)}")

# Обновляем статусы
if len(a_broken) > 0:
    df_a.loc[a_broken, 'status'] = 'dropped'
    df_a.loc[a_broken, 'drop_reason'] = 'broken_wkt'
if len(b_broken) > 0:
    df_b.loc[b_broken, 'status'] = 'dropped'
    df_b.loc[b_broken, 'drop_reason'] = 'broken_wkt'

# Обновляем статистику
stats['broken_wkt'] = [len(a_broken), len(b_broken)]

# Удаляем битые объекты
df_a = df_a[df_a['geometry'].notna()].copy()
df_b = df_b[df_b['geometry'].notna()].copy()
stats['records_kept'] = [len(df_a), len(df_b)]


print("Пример геометрии из А:")
print(df_a['geometry'].iloc[0])
print("\nПример геометрии из Б:")
print(df_b['geometry'].iloc[0])

print("\nОбновленная статитистика")
print(pd.DataFrame(stats))

A: успешно 171454 из 171454
B: успешно 161076 из 161076

Битых WKT в А: 0
Битых WKT в Б: 0
Пример геометрии из А:
MULTIPOLYGON (((30.10095 59.82052, 30.10085 59.82046, 30.10078 59.82048, 30.10076 59.82048, 30.10083 59.82054, 30.10076 59.82056, 30.10078 59.82058, 30.10095 59.82052)), ((30.10074 59.82045, 30.10065 59.82049, 30.10068 59.82051, 30.10076 59.82048, 30.10074 59.82045)))

Пример геометрии из Б:
MULTIPOLYGON (((30.095068678257853 59.84887228683547, 30.095125910806065 59.848711780942246, 30.09513938097476 59.848330505945306, 30.095143873837415 59.848305689193374, 30.09515285563249 59.84828538416975, 30.095152854737673 59.84818611663943, 30.095139380381173 59.848163555659724, 30.095139379342836 59.84813196998836, 30.09513488895632 59.848125202045, 30.095121413063 59.84811392193859, 30.095121414110974 59.84809361632416, 30.095107938550072 59.84808008004143, 30.09510344754537 59.848077824447074, 30.095085481351212 59.84807331168841, 30.095013615676095 59.848073311862336, 30.0949058

## Этап 3. Проверка валидности геометрии

Даже успешно распарсенные полигоны могут быть невалидными:
- самопересекающиеся границы (polygon в форме "бабочки")
- разрывы в контуре
- вырожденные полигоны (нулевая площадь)

Для проверки используется метод `is_valid` библиотеки GeoPandas. Если полигон невалиден, мы пытаемся его исправить:
1. `buffer(0)` — часто исправляет самопересечения
2. `make_valid()` — более мощный метод исправления

Если исправление удалось, объект получает статус `repaired` и флаг `repaired_geometry`.
Если исправить не удалось — объект удаляется с причиной `invalid_geometry`.

**Результат:**
- Невалидных полигонов (самопересечения, разрывы) не обнаружено
- Пустых геометрий нет
- Геометрия прошла базовую проверку топологии

In [17]:
# Создаём GeoDataFrame
gdf_a = gpd.GeoDataFrame(df_a, geometry='geometry', crs='EPSG:4326')
gdf_b = gpd.GeoDataFrame(df_b, geometry='geometry', crs='EPSG:4326')

# Проверяем валидность и пустоту
a_invalid = ~gdf_a.geometry.is_valid
b_invalid = ~gdf_b.geometry.is_valid
a_empty = gdf_a.geometry.is_empty
b_empty = gdf_b.geometry.is_empty

# Обновляем статистику
stats['empty_geometry'] = [a_empty.sum(), b_empty.sum()]

# Функция для исправления геометрии
def repair_geometry(geom):
    if geom is None or geom.is_valid:
        return geom
    try:
        repaired = geom.buffer(0)
        if repaired.is_valid:
            return repaired
        repaired = make_valid(geom)
        if repaired.is_valid:
            return repaired
    except:
        pass
    return None

# Исправляем невалидные полигоны
repaired_a = 0
if a_invalid.sum() > 0:
    gdf_a.loc[a_invalid, 'geometry'] = gdf_a.loc[a_invalid, 'geometry'].apply(repair_geometry)
    repaired_a = (~gdf_a.loc[a_invalid, 'geometry'].is_valid).sum()
    repaired_a = a_invalid.sum() - repaired_a

repaired_b = 0
if b_invalid.sum() > 0:
    gdf_b.loc[b_invalid, 'geometry'] = gdf_b.loc[b_invalid, 'geometry'].apply(repair_geometry)
    repaired_b = (~gdf_b.loc[b_invalid, 'geometry'].is_valid).sum()
    repaired_b = b_invalid.sum() - repaired_b

stats['repaired_geometry'] = [repaired_a, repaired_b]

a_still_invalid = gdf_a[~gdf_a.geometry.is_valid].index
b_still_invalid = gdf_b[~gdf_b.geometry.is_valid].index

if len(a_still_invalid) > 0:
    gdf_a.loc[a_still_invalid, 'status'] = 'dropped'
    gdf_a.loc[a_still_invalid, 'drop_reason'] = 'invalid_geometry'

if len(b_still_invalid) > 0:
    gdf_b.loc[b_still_invalid, 'status'] = 'dropped'
    gdf_b.loc[b_still_invalid, 'drop_reason'] = 'invalid_geometry'

# Удаляем из датасетов
gdf_a = gdf_a[gdf_a.geometry.is_valid].copy()
gdf_b = gdf_b[gdf_b.geometry.is_valid].copy()

stats['records_kept'] = [len(gdf_a), len(gdf_b)]


print(f"A: осталось {len(gdf_a)} записей")
print(f"B: осталось {len(gdf_b)} записей")

print("Статистико")
print(pd.DataFrame(stats))

A: осталось 171454 записей
B: осталось 161076 записей
Статистико
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  
0                  0                      0                  0        171454  
1                  0                      0                  0        161076  


## Шаг 4 Проверка вырожденных отверстий (invalid_holes)

Помимо базовой валидности (is_valid), полигоны могут иметь проблемы с внутренними отверстиями (дырками). Вырожденные отверстия — это внутренние контуры, которые:
- не замкнуты (not is_ring)
- или пересекаются с внешним контуром (crosses)

Такие объекты могут вызывать ошибки при пространственных операциях (пересечения, объединения). Они удаляются во избежание проблем при сопоставлении источников.


Расхождение в количестве полигонов и отверстий означает, что в источнике Б один полигон содержал два вырожденных отверстия.

**Результат:**
- Удалено из А: 8 объектов
- Удалено из Б: 7 объектов
- После удаления: А — 171 446 записей, Б — 161 069 записей


In [18]:
def check_invalid_holes_detailed(gdf, name):
    bad_polygons = []
    bad_holes_count = 0

    for idx, geom in gdf.geometry.items():
        if geom is None or geom.geom_type != "Polygon":
            continue
        exterior = geom.exterior
        poly_has_bad = False
        for interior in geom.interiors:
            if not interior.is_ring or interior.crosses(exterior):
                bad_holes_count += 1
                poly_has_bad = True
        if poly_has_bad:
            bad_polygons.append(idx)

    return bad_polygons, bad_holes_count

# Находим объекты с вырожденными отверстиями
a_bad_polygons, a_bad_holes = check_invalid_holes_detailed(gdf_a, "A")
b_bad_polygons, b_bad_holes = check_invalid_holes_detailed(gdf_b, "B")

print(f"A: полигонов с вырожденными отверстиями: {len(a_bad_polygons)}")
print(f"A: всего вырожденных отверстий: {a_bad_holes}")
print(f"B: полигонов с вырожденными отверстиями: {len(b_bad_polygons)}")
print(f"B: всего вырожденных отверстий: {b_bad_holes}")

# Помечаем и удаляем полигоны
if len(a_bad_polygons) > 0:
    gdf_a.loc[a_bad_polygons, 'status'] = 'dropped'
    gdf_a.loc[a_bad_polygons, 'drop_reason'] = 'invalid_holes'
    gdf_a = gdf_a.drop(index=a_bad_polygons).copy()
    print(f"\nУдалено из А: {len(a_bad_polygons)} объектов")

if len(b_bad_polygons) > 0:
    gdf_b.loc[b_bad_polygons, 'status'] = 'dropped'
    gdf_b.loc[b_bad_polygons, 'drop_reason'] = 'invalid_holes'
    gdf_b = gdf_b.drop(index=b_bad_polygons).copy()
    print(f"\nУдалено из Б: {len(b_bad_polygons)} объектов")

# Обновляем статистику
stats['records_kept'] = [len(gdf_a), len(gdf_b)]


print(f"A: {len(gdf_a)} записей")
print(f"B: {len(gdf_b)} записей")
print(pd.DataFrame(stats))

A: полигонов с вырожденными отверстиями: 8
A: всего вырожденных отверстий: 8
B: полигонов с вырожденными отверстиями: 7
B: всего вырожденных отверстий: 9

Удалено из А: 8 объектов

Удалено из Б: 7 объектов
A: 171446 записей
B: 161069 записей
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  
0                  0                      0                  0        171446  
1                  0                      0                  0        161069  


## Этап 5. Перевод в метрическую проекцию и расчёт площади

Для корректного расчёта площади и расстояний необходимо перевести геометрию из градусов (EPSG:4326) в метрическую проекцию (EPSG:3857). После перевода вычисляется площадь каждого полигона и сохраняется в колонку `geom_area_m2`.
**Важное наблюдение**
Геометрическая площадь (`geom_area_m2`) оказалась примерно в 3 раза больше атрибутивной (`area_sq_m`). Это системное расхождение, связанное с разными методами расчёта. Для моделирования будем использовать геометрическую площадь.
**Результат:**
- Площадь рассчитана для всех объектов
- Минимальная площадь: А — 1.2 м², Б — 1.3 м²
- Максимальная площадь: А — 547 379 м², Б — 287 641 м²

In [19]:
gdf_a = gdf_a.to_crs('EPSG:3857')
gdf_b = gdf_b.to_crs('EPSG:3857')

gdf_a['geom_area_m2'] = gdf_a.geometry.area
gdf_b['geom_area_m2'] = gdf_b.geometry.area

print(f"A: {len(gdf_a)} записей, площадь от {gdf_a['geom_area_m2'].min():.1f} до {gdf_a['geom_area_m2'].max():.1f} м²")
print(f"B: {len(gdf_b)} записей, площадь от {gdf_b['geom_area_m2'].min():.1f} до {gdf_b['geom_area_m2'].max():.1f} м²")

print(pd.DataFrame(stats))

A: 171446 записей, площадь от 1.2 до 547379.7 м²
B: 161069 записей, площадь от 1.3 до 287641.7 м²
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  
0                  0                      0                  0        171446  
1                  0                      0                  0        161069  


## Этап 6. Удаление маленьких объектов (<15 м²)
Удаляем объекты с геометрической площадью <15 м². Такие объекты являются мусором: сараи, гаражи, мусорные площадки, теплицы (в Б).

**Почему 15 м², а не 5 м²?**
Геометрическая площадь в 3 раза больше атрибутивной. Объекты с атрибутивной площадью <4 м² имеют реальную площадь <15 м². Порог 15 м² гарантирует удаление только явного мусора.

**Результат:**
- А: удалено 1 929 объектов
- Б: удалено 9 объектов
- Минимальная площадь после удаления: А — 16.0 м², Б — 15.0 м²

In [20]:
# Проверка текущего количества
before_a = len(gdf_a)
before_b = len(gdf_b)
print(f"A: {before_a} записей")
print(f"B: {before_b} записей")

small_a = gdf_a[gdf_a['geom_area_m2'] < 15]
small_b = gdf_b[gdf_b['geom_area_m2'] < 15]
print(f"A: объектов < 15 м²: {len(small_a)}")
print(f"B: объектов < 15 м²: {len(small_b)}")

# Помечаем удаляемые объекты
gdf_a.loc[small_a.index, 'status'] = 'dropped'
gdf_a.loc[small_a.index, 'drop_reason'] = 'small_utility_object'

gdf_b.loc[small_b.index, 'status'] = 'dropped'
gdf_b.loc[small_b.index, 'drop_reason'] = 'small_utility_object'

# Удаляем
gdf_a = gdf_a[gdf_a['geom_area_m2'] >= 15].copy()
gdf_b = gdf_b[gdf_b['geom_area_m2'] >= 15].copy()


print(f"A: было {before_a}, стало {len(gdf_a)} (удалено {before_a - len(gdf_a)})")
print(f"B: было {before_b}, стало {len(gdf_b)} (удалено {before_b - len(gdf_b)})")

# Обновляем статистику
stats['small_utility_dropped'] = [before_a - len(gdf_a), before_b - len(gdf_b)]
stats['records_kept'] = [len(gdf_a), len(gdf_b)]

# Проверка минимальной площади после удаления
print(f"A: мин. площадь = {gdf_a['geom_area_m2'].min():.1f} м²")
print(f"B: мин. площадь = {gdf_b['geom_area_m2'].min():.1f} м²")


print(pd.DataFrame(stats))

A: 171446 записей
B: 161069 записей
A: объектов < 15 м²: 1929
B: объектов < 15 м²: 9
A: было 171446, стало 169517 (удалено 1929)
B: было 161069, стало 161060 (удалено 9)
A: мин. площадь = 16.0 м²
B: мин. площадь = 15.0 м²
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  
0                  0                   1929                  0        169517  
1                  0                      9                  0        161060  


## Этап 7. Исправление логических ошибок в атрибутах (А)

В источнике А обнаружено 420 записей, где минимальная этажность превышает максимальную (`gkh_floor_count_min > gkh_floor_count_max`). Это физически невозможно, поэтому значения меняются местами.

Объекты, в которых обнаружена и исправлена ошибка, получают статус `repaired` и флаг `swapped_min_max`.

**Результат:**
- Исправлено 420 записей
- Ошибок больше нет

In [21]:
# Поиск записей с min > max
mask = (gdf_a['gkh_floor_count_min'].notna() &
        gdf_a['gkh_floor_count_max'].notna() &
        (gdf_a['gkh_floor_count_min'] > gdf_a['gkh_floor_count_max']))

if mask.sum() > 0:
    # Сохраняем индексы исправленных записей
    repaired_indices = gdf_a[mask].index.tolist()

    # Меняем местами
    gdf_a.loc[mask, ['gkh_floor_count_min', 'gkh_floor_count_max']] = \
        gdf_a.loc[mask, ['gkh_floor_count_max', 'gkh_floor_count_min']].values

    # Обновляем статус для исправленных объектов
    gdf_a.loc[repaired_indices, 'status'] = 'repaired'
    gdf_a.loc[repaired_indices, 'quality_flags'] = gdf_a.loc[repaired_indices, 'quality_flags'] + 'swapped_min_max;'


    # Проверяем, что ошибок больше нет
    mask_after = (gdf_a['gkh_floor_count_min'].notna() &
                  gdf_a['gkh_floor_count_max'].notna() &
                  (gdf_a['gkh_floor_count_min'] > gdf_a['gkh_floor_count_max']))


# Обновляем статистику (считаем исправленные)
stats['repaired_count'] = [len(repaired_indices) if mask.sum() > 0 else 0, 0]


print(f"A: после исправления")

print(pd.DataFrame(stats))

A: после исправления
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  \
0                  0                   1929                  0        169517   
1                  0                      9                  0        161060   

   repaired_count  
0             420  
1               0  


## Этап 8. Обработка `avg_floor_height = 0` (Б)

В источнике Б обнаружено 4 260 записей с `avg_floor_height = 0`. Это не физическая высота этажа, а заглушка для пропущенных данных. Значения заменяются на медиану по типу здания (`purpose_of_building`). Если для типа здания нет данных, используется общая медиана.

Объекты, в которых пропуск был заполнен, получают статус `repaired` и флаг `filled_avg_floor_height`.

**Результат:**
- Заполнено 4 260 записей
- Нулевых значений не осталось
- Распределение `avg_floor_height` стало физически обоснованным (медиана 3.0 м)

In [22]:
# Поиск записей с avg_floor_height = 0
zero_mask = (gdf_b['avg_floor_height'] == 0)

if zero_mask.sum() > 0:
    median_by_purpose = gdf_b.groupby('purpose_of_building')['avg_floor_height'].median()
    # Заполняем медианой по типу здания
    repaired_indices = []

    for idx in gdf_b[zero_mask].index:
        purpose = gdf_b.loc[idx, 'purpose_of_building']
        if pd.notna(purpose) and purpose in median_by_purpose.index:
            new_value = median_by_purpose[purpose]
        else:
            new_value = gdf_b['avg_floor_height'].median()

        gdf_b.loc[idx, 'avg_floor_height'] = new_value
        repaired_indices.append(idx)

    # Обновляем статус для исправленных объектов
    gdf_b.loc[repaired_indices, 'status'] = 'repaired'
    gdf_b.loc[repaired_indices, 'quality_flags'] = gdf_b.loc[repaired_indices, 'quality_flags'] + 'filled_avg_floor_height;'


    # Проверяем, что нулей не осталось
    zero_mask_after = (gdf_b['avg_floor_height'] == 0)
    print(f"Осталось нулей: {zero_mask_after.sum()}")
else:
    print("Нулевых значений не найдено")

# Обновляем статистику (считаем исправленные в Б)
stats['repaired_count'] = [stats['repaired_count'][0], len(repaired_indices) if zero_mask.sum() > 0 else 0]


print(gdf_b['avg_floor_height'].describe())

print(pd.DataFrame(stats))

Осталось нулей: 0
count    161060.000000
mean          3.452341
std           0.499133
min           2.000000
25%           3.000000
50%           3.000000
75%           4.000000
max          13.000000
Name: avg_floor_height, dtype: float64
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  \
0                  0                   1929                  0        169517   
1                  0                      9                  0        161060   

   repaired_count  
0             420  
1            4258  


## Этап 9. Удаление выбросов в источнике Б

В источнике Б выявлены аномальные значения, которые необходимо удалить:

| Тип выброса | Количество | Причина удаления |
|-------------|------------|------------------|
| `stairs <= 0` | 17 | Этажность не может быть ≤ 0 |
| `avg_floor_height > 10` | 2 | Слишком высоко для средней высоты этажа |
| `height` пропуски | 66 | Нет данных о высоте |

Объекты с высотой >300 м (Лахта-Центр) **не удаляются**, так как это реальное здание.

**Результат:**
- Всего удалено 85 объектов в Б
- Ключевые поля без пропусков и выбросов

In [23]:
# Находим выбросы
stairs_bad = (gdf_b['stairs'] <= 0) & (gdf_b['stairs'].notna())
height_bad = (gdf_b['avg_floor_height'] > 10) & (gdf_b['avg_floor_height'].notna())
height_null = gdf_b['height'].isna()
bad_mask = stairs_bad | height_bad | height_null

# Помечаем удаляемые
if bad_mask.sum() > 0:
    gdf_b.loc[bad_mask, 'status'] = 'dropped'
    gdf_b.loc[stairs_bad & ~height_bad & ~height_null, 'drop_reason'] = 'stairs_invalid'
    gdf_b.loc[height_bad & ~stairs_bad & ~height_null, 'drop_reason'] = 'avg_floor_height_too_high'
    gdf_b.loc[height_null, 'drop_reason'] = 'missing_height'
    gdf_b.loc[stairs_bad & height_null, 'drop_reason'] = 'stairs_invalid;missing_height'
    gdf_b.loc[height_bad & height_null, 'drop_reason'] = 'avg_floor_height_too_high;missing_height'

    before = len(gdf_b)
    gdf_b = gdf_b[~bad_mask].copy()
    stats['records_kept'] = [len(gdf_a), len(gdf_b)]


print(f"A: {len(gdf_a)} записей")
print(f"B: {len(gdf_b)} записей")
print()
print(pd.DataFrame(stats))

A: 169517 записей
B: 160975 записей

  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  \
0                  0                   1929                  0        169517   
1                  0                      9                  0        160975   

   repaired_count  
0             420  
1            4258  


## Этап 10. Проверка согласованности данных

Для проверки качества данных рассчитываются две метрики:
- `height_per_floor = height / stairs` — высота одного этажа
- `height_error_abs = |height - stairs × avg_floor_height|` — ошибка между реальной и расчётной высотой

**Результаты:**
- `height_per_floor` (медиана): 3.3 м — соответствует строительным нормам
- `height_error_abs` (медиана): 0.5 м — данные хорошо согласованы

**Аномалии:**
- `height_per_floor < 2`: 8 объектов
- `height_per_floor > 6`: 14 объектов
- `height_error_abs > 10`: 35 объектов

Аномалии составляют **0.022%** от всех данных. Это крайне мало, поэтому они не удаляются, чтобы не терять потенциально полезную информацию.

In [24]:
# Создаём копию для проверки
b_check = gdf_b.copy()
mask = b_check['stairs'].notna() & b_check['height'].notna() & b_check['avg_floor_height'].notna()

# Считаем метрики
b_check.loc[mask, 'height_per_floor'] = b_check.loc[mask, 'height'] / b_check.loc[mask, 'stairs']
b_check.loc[mask, 'height_error_abs'] = abs(
    b_check.loc[mask, 'height'] - b_check.loc[mask, 'stairs'] * b_check.loc[mask, 'avg_floor_height']
)

# Находим аномалии
low_floor = (b_check['height_per_floor'] < 2).sum()
high_floor = (b_check['height_per_floor'] > 6).sum()
large_error = (b_check['height_error_abs'] > 10).sum()


print(f"Всего записей в Б: {len(gdf_b)}")
print(f"height_per_floor < 2: {low_floor}")
print(f"height_per_floor > 6: {high_floor}")
print(f"height_error_abs > 10: {large_error}")
print(f"\nАномалии составляют {large_error/len(gdf_b)*100:.3f}% от всех данных")

print(pd.DataFrame(stats)) # аномалий оч мало, можно оставить

Всего записей в Б: 160975
height_per_floor < 2: 8
height_per_floor > 6: 14
height_error_abs > 10: 35

Аномалии составляют 0.022% от всех данных
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  \
0                  0                   1929                  0        169517   
1                  0                      9                  0        160975   

   repaired_count  
0             420  
1            4258  


## Этап 11. Проверка на дубликаты

Проверка на полные дубликаты (идентичная WKT-строка) показала, что таких объектов нет ни в одном источнике.

Почти-дубликаты (геометрически похожие объекты) обнаружены только в источнике А — 46 объектов (23 пары). Они не удаляются, так как могут быть частями одного физического здания (дом с пристройкой, комплекс зданий). При сопоставлении источников эти объекты будут объединены в компоненты связности.

**Результат:**
- Полных дубликатов: 0
- Почти-дубликаты: 46 объектов в А (оставлены)

In [25]:
# Проверка полных дубликатов (по WKT)

# Преобразуем геометрию в WKT-строки для сравнения
wkt_a = gdf_a.geometry.apply(lambda g: g.wkt if g is not None else None)
wkt_b = gdf_b.geometry.apply(lambda g: g.wkt if g is not None else None)

full_dup_a = wkt_a.duplicated().sum()
full_dup_b = wkt_b.duplicated().sum()

print(f"A: полных дубликатов: {full_dup_a}")
print(f"B: полных дубликатов: {full_dup_b}")

# Если есть полные дубликаты — удаляем
if full_dup_a > 0:
    dup_mask_a = wkt_a.duplicated(keep='first')
    gdf_a.loc[dup_mask_a, 'status'] = 'dropped'
    gdf_a.loc[dup_mask_a, 'drop_reason'] = 'duplicate'
    gdf_a = gdf_a[~dup_mask_a].copy()
    print(f"Удалено полных дубликатов из А: {full_dup_a}")

if full_dup_b > 0:
    dup_mask_b = wkt_b.duplicated(keep='first')
    gdf_b.loc[dup_mask_b, 'status'] = 'dropped'
    gdf_b.loc[dup_mask_b, 'drop_reason'] = 'duplicate'
    gdf_b = gdf_b[~dup_mask_b].copy()
    print(f"Удалено полных дубликатов из Б: {full_dup_b}")

# Почти-дубликаты не удаляем, так как они могут быть частями одного здания (например, дом с пристройкой, комплекс зданий).
# При сопоставлении они объединятся в компоненты связности.


print(f"A: {len(gdf_a)} записей")
print(f"B: {len(gdf_b)} записей")
print(f"Почти-дубликаты в А (46 объектов) оставлены с пометкой для последующего объединения")


# Обновляем статистику
stats['duplicate_dropped'] = [full_dup_a, full_dup_b]
stats['records_kept'] = [len(gdf_a), len(gdf_b)]

print(pd.DataFrame(stats))

A: полных дубликатов: 0
B: полных дубликатов: 0
A: 169517 записей
B: 160975 записей
Почти-дубликаты в А (46 объектов) оставлены с пометкой для последующего объединения
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  \
0                  0                   1929                  0        169517   
1                  0                      9                  0        160975   

   repaired_count  
0             420  
1            4258  


## Этап 12. Заполнение пропусков в ключевых полях (Б)

На даном этапе заполняются оставшиеся пропуски:

1. **`purpose_of_building`** — 3 964 пропуска (2.5%) заполняются значением `'unknown'`. Это позволяет сохранить все объекты и не терять данные.

2. **`stairs`** — 3 935 пропусков (2.5%) заполняются медианой по типу здания (`purpose_of_building`). Это логичный подход, так как разные типы зданий имеют разную типовую этажность.

**Результат:**
- Пропусков в ключевых полях не осталось
- Количество уникальных типов зданий: 18 (17 исходных + 'unknown')
- Статистика исправленных в Б обновлена

In [26]:
# Заполнение purpose_of_building
print("\n--- 1. purpose_of_building ---")
before_purpose = gdf_b['purpose_of_building'].isna().sum()
print(f"Пропусков: {before_purpose}")

if before_purpose > 0:
    gdf_b.loc[gdf_b['purpose_of_building'].isna(), 'purpose_of_building'] = 'unknown'
    gdf_b.loc[gdf_b['purpose_of_building'].isna(), 'status'] = 'repaired'
    gdf_b.loc[gdf_b['purpose_of_building'].isna(), 'quality_flags'] = 'filled_purpose;'
    print(f"Заполнено {before_purpose} пропусков значением 'unknown'")

after_purpose = gdf_b['purpose_of_building'].isna().sum()
print(f"Пропусков после заполнения: {after_purpose}")

# Заполнение stairs медианой по типу здания
print("\n--- 2. stairs (медиана по purpose_of_building) ---")
before_stairs = gdf_b['stairs'].isna().sum()
print(f"Пропусков stairs до заполнения: {before_stairs}")

if before_stairs > 0:
    # Считаем медиану для каждого типа здания
    median_by_purpose = gdf_b.groupby('purpose_of_building')['stairs'].median()

    # Создаём маску пропусков
    mask_stairs = gdf_b['stairs'].isna()
    repaired_indices = gdf_b[mask_stairs].index.tolist()

    # Заполняем медианой по типу здания
    for idx in repaired_indices:
        purpose = gdf_b.loc[idx, 'purpose_of_building']
        if pd.notna(purpose) and purpose in median_by_purpose.index:
            new_value = median_by_purpose[purpose]
        else:
            new_value = gdf_b['stairs'].median()
        gdf_b.loc[idx, 'stairs'] = new_value

    # Обновляем статус
    gdf_b.loc[repaired_indices, 'status'] = 'repaired'
    gdf_b.loc[repaired_indices, 'quality_flags'] = gdf_b.loc[repaired_indices, 'quality_flags'] + 'filled_stairs;'

    print(f"Заполнено {before_stairs} пропусков stairs")

after_stairs = gdf_b['stairs'].isna().sum()
print(f"Пропусков stairs после заполнения: {after_stairs}")

print(f"B: purpose_of_building пропусков: {gdf_b['purpose_of_building'].isna().sum()}")
print(f"B: stairs пропусков: {gdf_b['stairs'].isna().sum()}")
print(f"B: unique purpose_of_building: {gdf_b['purpose_of_building'].nunique()}")
print(f"  из них 'unknown': {(gdf_b['purpose_of_building'] == 'unknown').sum()}")

# Обновляем статистику
# repaired_count уже был 4258, добавляем заполненные
stats['repaired_count'][1] = stats['repaired_count'][1] + before_purpose + before_stairs

print(pd.DataFrame(stats))


--- 1. purpose_of_building ---
Пропусков: 3964
Заполнено 3964 пропусков значением 'unknown'
Пропусков после заполнения: 0

--- 2. stairs (медиана по purpose_of_building) ---
Пропусков stairs до заполнения: 3935
Заполнено 3935 пропусков stairs
Пропусков stairs после заполнения: 0
B: purpose_of_building пропусков: 0
B: stairs пропусков: 0
B: unique purpose_of_building: 18
  из них 'unknown': 3964
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  \
0                  0                   1929                  0        169517   
1                  0                      9                  0        160975   

   repaired_count  
0             420  
1           12157  


## Этап 13. Определение `is_relevant_building`

Создаётся колонка `is_relevant_building`, которая показывает, является ли объект полезным зданием для модели.

**`is_relevant_building = False` для:**
- Объектов, которые были удалены (статус `dropped`)
- Объектов, не являющихся зданиями: в Б это теплицы (`Парники, оранжереи, теплицы`)

Для всех остальных объектов значение `True`.

**Результат:**
- А: 169 525 релевантных зданий (100%)
- Б: 160 954 релевантных здания (99.98%)
- Нерелевантными оказались 28 теплиц в Б

In [27]:
# Добавляем колонку
gdf_a['is_relevant_building'] = True
gdf_b['is_relevant_building'] = True

# Объекты с status = dropped
dropped_a = gdf_a[gdf_a['status'] == 'dropped']
dropped_b = gdf_b[gdf_b['status'] == 'dropped']
if len(dropped_a) > 0:
    gdf_a.loc[dropped_a.index, 'is_relevant_building'] = False
if len(dropped_b) > 0:
    gdf_b.loc[dropped_b.index, 'is_relevant_building'] = False

# Объекты, не являющиеся зданиями (теплицы)
greenhouses = gdf_b[gdf_b['purpose_of_building'] == 'Парники, оранжереи, теплицы']
if len(greenhouses) > 0:
    gdf_b.loc[greenhouses.index, 'is_relevant_building'] = False


print(f"A: релевантных зданий: {gdf_a['is_relevant_building'].sum()} / {len(gdf_a)}")
print(f"B: релевантных зданий: {gdf_b['is_relevant_building'].sum()} / {len(gdf_b)}")

print(pd.DataFrame(stats))

A: релевантных зданий: 169517 / 169517
B: релевантных зданий: 160947 / 160975
  source  initial_count  broken_wkt  empty_geometry  repaired_geometry  \
0      A         171454           0               0                  0   
1      B         161076           0               0                  0   

   zero_area_dropped  small_utility_dropped  duplicate_dropped  records_kept  \
0                  0                   1929                  0        169517   
1                  0                      9                  0        160975   

   repaired_count  
0             420  
1           12157  


## Этап 13. Итоговая статистика очистки

После выполнения всех этапов очистки сформирована итоговая статистика, отражающая все изменения, произошедшие с данными.

**Основные показатели:**

| Показатель | А | Б |
|------------|---|---|
| Исходное количество записей | 171 454 | 161 076 |
| Удалено (invalid_holes) | 8 | 7 |
| Удалено (маленькие объекты <15 м²) | 1 929 | 9 |
| Удалено (stairs ≤ 0) | 0 | 17 |
| Удалено (avg_floor_height > 10) | 0 | 2 |
| Удалено (missing height) | 0 | 66 |
| **Всего удалено** | **1 937** | **101** |
| Исправлено (min > max в А) | 420 | 0 |
| Исправлено (avg_floor_height = 0) | 0 | 4 258 |
| Исправлено (purpose_of_building) | 0 | 3 964 |
| Исправлено (stairs) | 0 | 3 935 |
| **Всего исправлено** | **420** | **12 157** |
| **Осталось записей** | **169 517** | **160 975** |
| **Релевантных зданий** | **169 517** | **160 947** |

**Статусы объектов:**

- **А:** 169 097 объектов в статусе `clean`, 420 — `repaired`
- **Б:** 156 718 объектов в статусе `clean`, 4 257 — `repaired`

**Качество данных:**

Все ключевые поля, используемые для моделирования, не содержат пропусков:
- `tags` (А) — 0% пропусков
- `geom_area_m2` (А и Б) — 0% пропусков
- `purpose_of_building` (Б) — 0% пропусков (заполнено `'unknown'`)
- `stairs` (Б) — 0% пропусков (заполнено медианой по типу здания)
- `height` (Б) — 0% пропусков
- `avg_floor_height` (Б) — 0% пропусков

**Аномалии:**

При проверке согласованности данных (`height_per_floor`, `height_error_abs`) выявлено 35 аномалий, что составляет **0.022%** от всех данных. Аномалии не удаляются, так как их доля пренебрежимо мала и они не повлияют на качество модели.

**Вывод:**

Очистка данных завершена. Оба датасета подготовлены к следующему этапу — сопоставлению источников (объединению А и Б в единый датасет физических зданий).

In [28]:
print("="*70)
print("ИТОГОВАЯ СТАТИСТИКА ПОСЛЕ ВСЕХ ЭТАПОВ ОЧИСТКИ")
print("="*70)

print("\n--- 1. Количество записей ---")
print(f"A: {len(gdf_a)} записей (было 171454, удалено {171454 - len(gdf_a)})")
print(f"B: {len(gdf_b)} записей (было 161076, удалено {161076 - len(gdf_b)})")

print("\n--- 2. Статусы объектов ---")
print("\nA:")
print(gdf_a['status'].value_counts())
print("\nB:")
print(gdf_b['status'].value_counts())

print("\n--- 3. Пропуски в ключевых полях (которые будем использовать) ---")
print("\nA (пропуски):")
a_cols = ['tags', 'geom_area_m2']
for col in a_cols:
    miss = gdf_a[col].isna().sum()
    print(f"  {col}: {miss} ({miss/len(gdf_a)*100:.2f}%)")

print("\nB (пропуски):")
b_cols = ['purpose_of_building', 'stairs', 'height', 'avg_floor_height', 'geom_area_m2']
for col in b_cols:
    miss = gdf_b[col].isna().sum()
    print(f"  {col}: {miss} ({miss/len(gdf_b)*100:.2f}%)")

print("\n--- 4. Пропуски в полях, которые НЕ используем (для информации) ---")
print("\nA (не используем):")
a_skip = ['title', 'gkh_address', 'gkh_floor_count_min', 'gkh_floor_count_max']
for col in a_skip:
    if col in gdf_a.columns:
        miss = gdf_a[col].isna().sum()
        print(f"  {col}: {miss} ({miss/len(gdf_a)*100:.2f}%)")

print("\nB (не используем):")
b_skip = ['type_street', 'name_street', 'letter', 'fraction', 'housing', 'building', 'subject', 'district']
for col in b_skip:
    if col in gdf_b.columns:
        miss = gdf_b[col].isna().sum()
        print(f"  {col}: {miss} ({miss/len(gdf_b)*100:.2f}%)")

print("\n--- 5. Категориальные признаки ---")
print(f"A: уникальных tags: {gdf_a['tags'].nunique()}")
print(f"B: уникальных purpose_of_building: {gdf_b['purpose_of_building'].nunique()}")
print(f"   из них 'unknown': {(gdf_b['purpose_of_building'] == 'unknown').sum()}")

print("\n--- 6. Числовые признаки (статистика) ---")
print("\nA (geom_area_m2):")
print(gdf_a['geom_area_m2'].describe())

print("\nB (height):")
print(gdf_b['height'].describe())

print("\nB (stairs):")
print(gdf_b['stairs'].describe())

print("\n--- 7. is_relevant_building ---")
print(f"A: релевантных: {gdf_a['is_relevant_building'].sum()} / {len(gdf_a)}")
print(f"B: релевантных: {gdf_b['is_relevant_building'].sum()} / {len(gdf_b)}")


print(pd.DataFrame(stats))





summary = pd.DataFrame({
    'Показатель': [
        'Исходное количество записей',
        'Удалено (маленькие объекты <15 м²)',
        'Удалено (stairs <= 0)',
        'Удалено (avg_floor_height > 10)',
        'Удалено (missing height)',
        'Всего удалено',
        'Исправлено (min > max в А)',
        'Исправлено (avg_floor_height = 0)',
        'Исправлено (purpose_of_building)',
        'Исправлено (stairs)',
        'Всего исправлено',
        'Осталось записей',
        'Релевантных зданий'
    ],
    'А': [
        stats['initial_count'][0],
        stats['small_utility_dropped'][0],
        0,
        0,
        0,
        stats['small_utility_dropped'][0],
        stats['repaired_count'][0],
        0,
        0,
        0,
        stats['repaired_count'][0],
        stats['records_kept'][0],
        gdf_a['is_relevant_building'].sum()
    ],
    'Б': [
        stats['initial_count'][1],
        stats['small_utility_dropped'][1],
        17,
        2,
        66,
        stats['small_utility_dropped'][1] + 17 + 2 + 66,
        0,
        4258,
        3964,
        3935,
        stats['repaired_count'][1],
        stats['records_kept'][1],
        gdf_b['is_relevant_building'].sum()
    ]
})

print(summary.to_string(index=False))


ИТОГОВАЯ СТАТИСТИКА ПОСЛЕ ВСЕХ ЭТАПОВ ОЧИСТКИ

--- 1. Количество записей ---
A: 169517 записей (было 171454, удалено 1937)
B: 160975 записей (было 161076, удалено 101)

--- 2. Статусы объектов ---

A:
status
clean       169097
repaired       420
Name: count, dtype: int64

B:
status
clean       156733
repaired      4242
Name: count, dtype: int64

--- 3. Пропуски в ключевых полях (которые будем использовать) ---

A (пропуски):
  tags: 0 (0.00%)
  geom_area_m2: 0 (0.00%)

B (пропуски):
  purpose_of_building: 0 (0.00%)
  stairs: 0 (0.00%)
  height: 0 (0.00%)
  avg_floor_height: 0 (0.00%)
  geom_area_m2: 0 (0.00%)

--- 4. Пропуски в полях, которые НЕ используем (для информации) ---

A (не используем):
  title: 162245 (95.71%)
  gkh_address: 151280 (89.24%)
  gkh_floor_count_min: 152226 (89.80%)
  gkh_floor_count_max: 151887 (89.60%)

B (не используем):
  type_street: 72974 (45.33%)
  name_street: 72974 (45.33%)
  letter: 152246 (94.58%)
  fraction: 159877 (99.32%)
  housing: 130452 (81.04%)

## Этап 14. Сохранение данных

Очищенные данные сохраняются в нескольких форматах для разных целей:

| Формат | Назначение | Размер (А/Б) |
|--------|------------|--------------|
| `.gpkg` | Для открытия в QGIS, визуализации | 56 МБ / 139 МБ |
| `.parquet` | Быстрая загрузка в Python (сжатый) | 16 МБ / 36 МБ |
| `.csv` | Просмотр в Excel (без геометрии) | 18 МБ / 85 МБ |

Также сохраняется файл `cleaning_statistics.csv` с итоговой статистикой очистки.

In [29]:

# GeoPackage
gdf_a.to_file(OUTPUT_DIR / "A_clean_final1.gpkg", driver="GPKG")
gdf_b.to_file(OUTPUT_DIR / "B_clean_final1.gpkg", driver="GPKG")

# Parquet
gdf_a.to_parquet(OUTPUT_DIR / "A_clean_final1.parquet")
gdf_b.to_parquet(OUTPUT_DIR / "B_clean_final1.parquet")

# CSV без геометрии
gdf_a.drop(columns=["geometry"]).to_csv(
    OUTPUT_DIR / "A_clean_final1.csv", index=False
)
gdf_b.drop(columns=["geometry"]).to_csv(
    OUTPUT_DIR / "B_clean_final1.csv", index=False
)

# Сохраняем статистику
summary.to_csv(OUTPUT_DIR / 'cleaning_statistics.csv', index=False)

print(f"A: {len(gdf_a)} записей, колонки: {list(gdf_a.columns)}")
print(f"B: {len(gdf_b)} записей, колонки: {list(gdf_b.columns)}")

A: 169517 записей, колонки: ['id', 'title', 'tags', 'geometry', 'area_sq_m', 'gkh_address', 'gkh_floor_count_min', 'gkh_floor_count_max', 'status', 'quality_flags', 'drop_reason', 'geom_area_m2', 'is_relevant_building']
B: 160975 записей, колонки: ['subject', 'district', 'type', 'locality', 'type_street', 'name_street', 'number', 'letter', 'fraction', 'housing', 'building', 'purpose_of_building', 'stairs', 'avg_floor_height', 'height', 'wkt', 'id', 'status', 'quality_flags', 'drop_reason', 'geometry', 'geom_area_m2', 'is_relevant_building']
